# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest1"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

target = FQN('bookings')

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
# TODO
from pyspark.sql.functions import broadcast

data_1 = bookings_df.join(hotels_df, on="hotel_id", how="inner")
data_1.explain()
data_1.explain(True)

data_2 = bookings_df.join(broadcast(hotels_df), on="hotel_id")
data_2.explain()
data_2.explain(True)

# the first query is about inner join and the second query is about broadcasthashjoin which used inner join. The first query uses shuffle as it uses wide transformation while the 2nd one uses narrow transformation as the smaller tableis copied to different workers for communication because of which there is no shuffle.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [hotel_id#13959, booking_id#13957, customer_id#13958, booking_date#13960, city#13961, nights#13962, amount#13963, status#13964, hotel_name#13989, city#13990, category#13991, star_rating#13992]
         +- PhotonBroadcastHashJoin [hotel_id#13959], [hotel_id#13988], Inner, BuildRight, false, true, false
            :- PhotonFilter isnotnull(hotel_id#13959)
            :  +- PhotonRowToColumnar
            :     +- FileScan csv [booking_id#13957,customer_id#13958,hotel_id#13959,booking_date#13960,city#13961,nights#13962,amount#13963,status#13964] Batched: false, DataFilters: [isnotnull(hotel_id#13959)], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/staynest1/bookings.csv], PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_date:d

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
# TODO
bookings_df.write.mode('overwrite').format('delta').saveAsTable(target)
update_data = spark.sql(f"update bookings set status = 'completed' where status = 'pending' ")
delete_data = spark.sql(f"delete from bookings where status = 'cancelled'  ")

spark.sql(f"describe history bookings").select(
    "version", "timestamp", "operation"
).show(truncate=False)


+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|25     |2026-09-23 15:42:09|DELETE                           |
|24     |2026-09-23 15:42:06|UPDATE                           |
|23     |2026-09-23 15:42:02|CREATE OR REPLACE TABLE AS SELECT|
|22     |2026-09-22 18:03:51|RESTORE                          |
|21     |2026-09-22 18:02:36|RESTORE                          |
|20     |2026-09-22 18:01:08|RESTORE                          |
|19     |2026-09-22 17:59:32|RESTORE                          |
|18     |2026-09-22 17:56:32|RESTORE                          |
|17     |2026-09-22 17:55:15|RESTORE                          |
|16     |2026-09-22 17:47:47|OPTIMIZE                         |
|15     |2026-09-22 17:47:44|DELETE                           |
|14     |2026-09-22 17:47:42|UPDATE                           |
|13     |2026-09-22 17:47:39|CREATE OR R

## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
# TODO
v0 = spark.read.format('delta').option('versionAsOf', 0).table(target)
v0.count()

current_version = spark.table(target)
current_count = current_version.count()

spark.sql(f"RESTORE TABLE {target} TO VERSION AS OF 0")
restored = spark.table(target)
restored.count()

display(spark.sql(f"DESCRIBE HISTORY {target}").select("version", "timestamp", "operation"))

version,timestamp,operation
27,2026-09-23T15:42:30.000Z,RESTORE
26,2026-09-23T15:42:13.000Z,OPTIMIZE
25,2026-09-23T15:42:09.000Z,DELETE
24,2026-09-23T15:42:06.000Z,UPDATE
23,2026-09-23T15:42:02.000Z,CREATE OR REPLACE TABLE AS SELECT
22,2026-09-22T18:03:51.000Z,RESTORE
21,2026-09-22T18:02:36.000Z,RESTORE
20,2026-09-22T18:01:08.000Z,RESTORE
19,2026-09-22T17:59:32.000Z,RESTORE
18,2026-09-22T17:56:32.000Z,RESTORE


## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
# TODO
result = spark.sql(f"optimize {target}")

result1 = spark.sql(f"optimize {target} ZORDER BY (city)")

# Optimize compacts small files into large ones.
# city column ha high cardinality while status has low cardinality. (Cardinality - Unique values in a column) 


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
# TODO
from pyspark.sql.functions import current_timestamp
bookings_df = read_csv("bookings")
bronze_bookings_data = bookings_df.withColumn('ingested_at', current_timestamp())
bronze_bookings_data.write.mode('overwrite').format('delta').option('overwriteSchema', True).saveAsTable('bronze_bookings')
bronze_bookings_data.count()


12000

## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
# TODO
from pyspark.sql.functions import col
silver_bookings_data = spark.table('bronze_bookings').filter(col('status') == 'completed').join(hotels_df.alias('h'), on='hotel_id', how = 'inner').drop(col('h.city')).select('*')
silver_bookings_data.show(5)
silver_bookings_data.write.mode('overwrite').format('delta').saveAsTable('silver_bookings')
spark.table('silver_bookings').show(5)


+--------+----------+-----------+------------+---------+------+--------+---------+--------------------+----------------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|     city|nights|  amount|   status|         ingested_at|      hotel_name|category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+--------------------+----------------+--------+-----------+
|    3095|   9000000|     701600|  2025-11-27|   Jaipur|     4| 6087.65|completed|2026-09-23 15:43:...|   Orchid Suites|  Budget|        3.8|
|    3112|   9000003|     700867|  2025-03-22|Bengaluru|     5| 7880.62|completed|2026-09-23 15:43:...|      Grand Stay|  Budget|        3.6|
|    3012|   9000006|     701336|  2025-11-25|    Delhi|     7|70999.15|completed|2026-09-23 15:43:...|Orchid Residency|  Luxury|        4.1|
|    3127|   9000007|     700868|  2025-04-10|   Mumbai|     2|15693.64|completed|2026-09-23 15:43:...|     Summit Stay|  Luxury|        4.5|
|    3

## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
# TODO
from pyspark.sql.functions import sum,count

gold_bookings_data = spark.table('silver_bookings').groupby(col('city')).agg(sum(col('amount')).alias('revenue'), count(col('booking_id')).alias('booking_count')).orderBy(col('revenue').desc())

gold_bookings_data.write.mode('overwrite').format('delta').saveAsTable('gold_city_revenue')
spark.table('gold_city_revenue').show(5)



+---------+--------------------+-------------+
|     city|             revenue|booking_count|
+---------+--------------------+-------------+
|      Goa|       4.459670179E7|         2546|
|   Mumbai| 3.624122111999999E7|         1715|
|    Delhi| 2.631428154000002E7|         1174|
|   Jaipur|2.4436853129999984E7|          979|
|Bengaluru|2.2670136969999984E7|         1318|
+---------+--------------------+-------------+
only showing top 5 rows


## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
# TODO
count_before_merge = spark.table('silver_bookings').count()
print(f"{count_before_merge} rows brfore merge" )

updates_df.createOrReplaceTempView('updates')

spark.sql(f"""
MERGE INTO {FQN('silver_bookings')} AS target
USING updates AS source
ON target.booking_id = source.booking_id

WHEN MATCHED THEN
    UPDATE SET
        target.hotel_id = source.hotel_id,
        target.customer_id = source.customer_id,
        target.booking_date = source.booking_date,
        target.city = source.city,
        target.nights = source.nights,
        target.amount = source.amount,
        target.status = source.status

WHEN NOT MATCHED THEN
    INSERT (
        booking_id,
        customer_id,
        hotel_id,
        booking_date,
        city,
        nights,
        amount,
        status,
        ingested_at
    )
    VALUES (
        source.booking_id,
        source.customer_id,
        source.hotel_id,
        source.booking_date,
        source.city,
        source.nights,
        source.amount,
        source.status,
        current_timestamp()
    )
""")

count_after_merge = spark.table('silver_bookings').count()
print(f"{count_after_merge} rows after merge" )



9660 rows brfore merge
9746 rows after merge
